# 实验一：Kaggle PM2.5 预测



1. 读取 `train.csv` 和 `test.csv`，若提供了 `sampleSubmission.csv` 则同时读取；
2. 使用连续 9 小时、18 项空气指标预测下一小时 PM2.5；
3. 标准化特征并用 Adagrad 训练线性回归模型；
4. 生成 `/kaggle/working/submit.csv` 

## 0. 导入依赖并定位竞赛数据



In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

INPUT_ROOT = Path('/kaggle/input')

print('Kaggle Input 中的文件：')
for path in sorted(INPUT_ROOT.rglob('*')):
    if path.is_file():
        print(path)

DATA_DIR = None
for train_candidate in INPUT_ROOT.rglob('train.csv'):
    candidate_dir = train_candidate.parent
    if (candidate_dir / 'test.csv').exists():
        DATA_DIR = candidate_dir
        break

assert DATA_DIR is not None, (
    '没有找到同时包含 train.csv 和 test.csv 的目录。'
    '请先在右侧 Add Input 中添加课程竞赛数据。'
)

TRAIN_PATH = DATA_DIR / 'train.csv'
TEST_PATH = DATA_DIR / 'test.csv'
SAMPLE_PATH = DATA_DIR / 'sampleSubmission.csv'

print('使用的数据目录：', DATA_DIR)
print('是否提供样例提交文件：', SAMPLE_PATH.exists())

Kaggle Input 中的文件：
/kaggle/input/competitions/2026zjutest1/test.csv
/kaggle/input/competitions/2026zjutest1/train.csv
使用的数据目录： /kaggle/input/competitions/2026zjutest1
是否提供样例提交文件： False


## 1. 读取并检查训练数据

训练集前三列通常是日期、站点和检测项目，后面 24 列是一天内逐小时的观测值。`NR` 和无法解析的值统一按 0 处理。

In [2]:
train_df = pd.read_csv(TRAIN_PATH, encoding='gb18030')

print('原始训练集形状：', train_df.shape)
display(train_df.head())

assert train_df.shape[1] >= 4, 'train.csv 的列数不符合课件数据格式。'

feature_names = train_df.iloc[:18, 2].astype(str)
pm25_matches = np.flatnonzero(
    feature_names.str.contains('PM2.5', case=False, regex=False).to_numpy()
)
PM25_INDEX = int(pm25_matches[0]) if len(pm25_matches) else 9
print('PM2.5 在每组18项指标中的位置：', PM25_INDEX)

train_num = (
    train_df.iloc[:, 3:]
    .replace('NR', 0)
    .apply(pd.to_numeric, errors='coerce')
    .to_numpy(dtype=float)
)
train_num = np.nan_to_num(train_num, nan=0.0)

assert train_num.shape[1] == 24, '训练集数值部分应当包含24个小时。'
assert train_num.shape[0] % (12 * 18) == 0, '训练集行数无法按12个月、每组18项指标拆分。'

DAYS_PER_MONTH = train_num.shape[0] // (12 * 18)
HOURS_PER_MONTH = DAYS_PER_MONTH * 24

print('每月天数：', DAYS_PER_MONTH)
print('每月小时数：', HOURS_PER_MONTH)

原始训练集形状： (3240, 27)


,日期,測站,測項,0,1,2,3,4,5,6,...,14,15,16,17,18,19,20,21,22,23
0,2014/1/1,豐原,AMB_TEMP,14,14,14,13,12,12,12,...,22,22,21,19,17,16,15,15,15,15
1,2014/1/1,豐原,CH4,1.8,1.8,1.8,1.8,1.8,1.8,1.8,...,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8
2,2014/1/1,豐原,CO,0.51,0.41,0.39,0.37,0.35,0.3,0.37,...,0.37,0.37,0.47,0.69,0.56,0.45,0.38,0.35,0.36,0.32
3,2014/1/1,豐原,NMHC,0.2,0.15,0.13,0.12,0.11,0.06,0.1,...,0.1,0.13,0.14,0.23,0.18,0.12,0.1,0.09,0.1,0.08
4,2014/1/1,豐原,NO,0.9,0.6,0.5,1.7,1.8,1.5,1.9,...,2.5,2.2,2.5,2.3,2.1,1.9,1.5,1.6,1.8,1.5


PM2.5 在每组18项指标中的位置： 9
每月天数： 15
每月小时数： 360


## 2. 构造 9 小时滑动窗口

每个样本包含 18 项指标过去 9 小时的数据，因此特征数是 $18\times9=162$；标签是紧接着的下一小时 PM2.5。课件数据每月 15 天，因此预计得到 $12\times351=4212$ 个训练样本。

In [3]:
month_data = np.empty((12, 18, HOURS_PER_MONTH), dtype=float)

for month in range(12):
    for day in range(DAYS_PER_MONTH):
        begin = (month * DAYS_PER_MONTH + day) * 18
        end = begin + 18
        month_data[month, :, day * 24:(day + 1) * 24] = train_num[begin:end, :]

WINDOW = 9
x_list = []
y_list = []

for month in range(12):
    for start in range(HOURS_PER_MONTH - WINDOW):
        feature = month_data[month, :, start:start + WINDOW].reshape(-1)
        target = month_data[month, PM25_INDEX, start + WINDOW]
        x_list.append(feature)
        y_list.append(target)

x = np.asarray(x_list, dtype=float)
y = np.asarray(y_list, dtype=float).reshape(-1, 1)
x = np.nan_to_num(x, nan=0.0)
y = np.nan_to_num(y, nan=0.0)

assert x.shape[1] == 18 * 9
assert len(x) == len(y)

print('训练特征形状：', x.shape)
print('训练标签形状：', y.shape)
print('课件数据的预期形状：(4212, 162) 和 (4212, 1)')

训练特征形状： (4212, 162)
训练标签形状： (4212, 1)
课件数据的预期形状：(4212, 162) 和 (4212, 1)


## 3. 标准化并训练线性回归模型

只使用训练集计算均值和标准差；测试集必须复用同一组统计量。模型按照课件使用 Adagrad 优化，并将权重保存到 `/kaggle/working/weight.npy`。

In [4]:
mean_x = np.mean(x, axis=0)
std_x = np.std(x, axis=0)
valid_std = std_x != 0

x_standardized = x.copy()
x_standardized[:, valid_std] = (
    x_standardized[:, valid_std] - mean_x[valid_std]
) / std_x[valid_std]

x_train = np.concatenate(
    [np.ones((len(x_standardized), 1)), x_standardized],
    axis=1,
)

w = np.zeros((x_train.shape[1], 1))
learning_rate = 100
iterations = 1000
adagrad = np.zeros_like(w)
eps = 1e-10

for iteration in range(iterations):
    error = x_train @ w - y
    loss = np.sqrt(np.mean(error ** 2))
    gradient = 2 * x_train.T @ error / len(x_train)
    adagrad += gradient ** 2
    w -= learning_rate * gradient / np.sqrt(adagrad + eps)

    if iteration % 100 == 0:
        print(f'{iteration}: RMSE={loss:.6f}')

assert np.isfinite(w).all(), '训练结果出现 NaN 或无穷值。'
np.save('/kaggle/working/weight.npy', w)
print('权重已保存：/kaggle/working/weight.npy')

0: RMSE=26.103713
100: RMSE=35.141762
200: RMSE=18.377660
300: RMSE=12.605111
400: RMSE=10.326857
500: RMSE=9.210694
600: RMSE=8.582850
700: RMSE=8.208529
800: RMSE=7.979155
900: RMSE=7.835748
权重已保存：/kaggle/working/weight.npy


## 4. 处理测试集并生成预测

测试集中每 18 行组成一个样本，每行包含连续 9 小时的数据。按照课件，本次应得到 60 条预测。

In [5]:
test_df = pd.read_csv(TEST_PATH, header=None, encoding='ISO-8859-1')

print('原始测试集形状：', test_df.shape)
display(test_df.head())

test_num = (
    test_df.iloc[:, 2:]
    .replace('NR', 0)
    .apply(pd.to_numeric, errors='coerce')
    .to_numpy(dtype=float)
)
test_num = np.nan_to_num(test_num, nan=0.0)

assert test_num.shape[1] == 9, '测试集数值部分应当包含连续9个小时。'
assert test_num.shape[0] % 18 == 0, '测试集行数不能按每个样本18项指标拆分。'

test_count = test_num.shape[0] // 18
test_x = np.empty((test_count, 18 * 9), dtype=float)

for i in range(test_count):
    test_x[i] = test_num[i * 18:(i + 1) * 18].reshape(-1)

test_x[:, valid_std] = (
    test_x[:, valid_std] - mean_x[valid_std]
) / std_x[valid_std]

test_x = np.concatenate(
    [np.ones((test_count, 1)), test_x],
    axis=1,
)
prediction = test_x @ w

assert np.isfinite(prediction).all(), '预测结果出现 NaN 或无穷值。'
print('预测数量：', len(prediction))
print('前5条预测：', prediction[:5].ravel())

原始测试集形状： (1080, 11)


,0,1,2,3,4,5,6,7,8,9,10
0,id_0,AMB_TEMP,20,20,19,18,16,15,14,14,13
1,id_0,CH4,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8,1.8
2,id_0,CO,0.27,0.26,0.3,0.48,0.49,0.42,0.36,0.32,0.28
3,id_0,NMHC,0.15,0.15,0.18,0.27,0.25,0.18,0.13,0.1,0.08
4,id_0,NO,1.6,1.7,1.4,1.3,1.2,1.2,0.9,0.9,1.1


预测数量： 60
前5条预测： [23.77782701 23.67305436 81.41437471 60.93256853 27.57638246]


## 5. 生成并检查 `submit.csv`

最终提交文件为 60 行、两列 `id` 和 `value`。

In [6]:
if SAMPLE_PATH.exists():
    submission = pd.read_csv(SAMPLE_PATH)
    assert submission.shape[1] == 2, 'sampleSubmission.csv 应当包含两列。'
    assert len(submission) == len(prediction), (
        f'样例提交有 {len(submission)} 行，但预测结果有 {len(prediction)} 行。'
    )
    submission[submission.columns[1]] = prediction.ravel()
else:
    submission = pd.DataFrame({
        'id': np.arange(1, len(prediction) + 1),
        'value': prediction.ravel(),
    })

OUTPUT_PATH = Path('/kaggle/working/submit.csv')
submission.to_csv(OUTPUT_PATH, index=False)

check = pd.read_csv(OUTPUT_PATH)
assert check.shape == submission.shape
assert check.iloc[:, 1].notna().all()
assert np.isfinite(check.iloc[:, 1].to_numpy(dtype=float)).all()

display(check.head())
print('提交文件位置：', OUTPUT_PATH)
print('提交文件形状：', check.shape)
print('提交文件列名：', check.columns.tolist())
print('缺失值数量：', check.isna().sum().to_dict())

,id,value
0,1,23.777827
1,2,23.673054
2,3,81.414375
3,4,60.932569
4,5,27.576382


提交文件位置： /kaggle/working/submit.csv
提交文件形状： (60, 2)
提交文件列名： ['id', 'value']
缺失值数量： {'id': 0, 'value': 0}
